In [18]:
# Utils
# ==============================================================================
import warnings

# Plot
# ==============================================================================
import matplotlib.pyplot as plt
import seaborn as sns

# Data
# ==============================================================================
import pandas as pd
from sklearn.datasets import make_classification

# Model
# ==============================================================================
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.tree import DecisionTreeClassifier

# Metrics
# ==============================================================================
from sklearn.metrics import accuracy_score

# **Info**
---

**@By**: Steven Bernal

**@Nickname**: Kaiziferr

**@Git**: https://github.com/Kaiziferr

# **Config**
---

In [19]:
sns.set(style="darkgrid")
warnings.simplefilter("ignore")
paleta = sns.color_palette("tab10").as_hex()
random_seed = 12354

# **Context**
---

A synthetic dataset was chosen for this classification problem because it provides a controlled and educational environment, specifically designed to explain the pre-pruning process in decision trees.

Working with real-world datasets usually requires a significant amount of effort in preprocessing tasks such as data cleaning, handling missing values, correcting inconsistencies, and transforming variables. While these steps are essential in any data science workflow, they are outside the scope of this discussion.

# **Goals**
---

- Implement pre-pruning in a decision tree applied to a classification problem using synthetic data.

# **Data**
---

The dataset is generated with 1,000 samples and 10 features. Of these, 6 features are informative and 2 are redundant. No duplicated features are included.

The task is a binary classification problem with two classes. Each class is distributed into two clusters within the feature space.

A label noise rate of 0.08 is introduced, meaning that 8% of the labels are randomly flipped. The class separation is set to 0.8, and the classes are balanced.

A random seed is used to ensure the reproducibility of the dataset.


In [20]:
X, y = make_classification(
    n_samples=1000,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    n_repeated=0,
    n_classes=2,
    n_clusters_per_class=2,
    weights=[0.5, 0.5],
    flip_y=0.08,
    class_sep=0.8,
    random_state=random_seed
)

In [21]:
X = pd.DataFrame(X, columns=[f'A{i}'for i in range(X.shape[1])])

# **Model**
---

The dataset is split into training and test sets using a random partition. 20% of the samples are assigned to the test set, while the remaining 80% are used for training.

The split preserves the original structure of the input features and their corresponding labels. A random seed is used to ensure the reproducibility of the partition.

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=random_seed
)

A parameter dictionary was defined to configure a decision tree model:

- The splitting criterion was set using the options gini, entropy, and log_loss, which determine the metric used to evaluate the quality of splits at each node.

- The split strategy was defined using best and random, controlling whether the optimal split is selected or a random one is chosen.

- The maximum depth of the tree was configured with values including None, 3, 5, 10, 20, and 30, thereby constraining the model’s growth at different levels of complexity.

- The minimum number of samples required to split an internal node was set to 2, 5, 10, and 20, defining the threshold at which a node is eligible for further partitioning.

- The minimum number of samples allowed in a leaf node was set to 1, 2, 4, and 8, controlling the smallest possible size of terminal nodes.

- The number of features considered at each split was defined using None, sqrt, log2, 0.5, and 0.75, thereby limiting or adjusting the subset of variables used for each decision split.

In [23]:
dic_params = {
    "criterion": ["gini", "entropy", "log_loss"],
    "splitter": ["best", "random"],
    "max_depth": [None, 3, 5, 10, 20, 30],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": [None, "sqrt", "log2", 0.5, 0.75]
}

- A classification model based on a decision tree was defined using DecisionTreeClassifier.

- A random seed was set through random_state to ensure the reproducibility of the model across different runs.

In [24]:
model_classifier_prepruning_v1 = DecisionTreeClassifier(random_state=random_seed)

Hyperparameter tuning was configured using cross-validated grid search via GridSearchCV:

- A decision tree model previously initialized was used as the base estimator. A parameter grid (param_grid) was defined to explore different model configurations.

- Accuracy was selected as the evaluation metric. Ten-fold cross-validation (cv=10) was applied, assessing model performance across different subsets of the training data.

- The refit=True option was enabled to retrain the model using the best combination of hyperparameters found during the search. Additionally, return_train_score=True was activated to store training performance metrics during the search process.

- Finally, the model was fitted using the training data X_train and y_train.

In [25]:
model_classifier_prepruning_v1 = GridSearchCV(
    estimator=model_classifier_prepruning_v1,
    param_grid=dic_params,
    scoring='accuracy',
    cv=10,
    refit=True,
    return_train_score = True
)
model_classifier_prepruning_v1.fit(X_train, y_train)

GridSearchCV(cv=10, estimator=DecisionTreeClassifier(random_state=12354),
             param_grid={'criterion': ['gini', 'entropy', 'log_loss'],
                         'max_depth': [None, 3, 5, 10, 20, 30],
                         'max_features': [None, 'sqrt', 'log2', 0.5, 0.75],
                         'min_samples_leaf': [1, 2, 4, 8],
                         'min_samples_split': [2, 5, 10, 20],
                         'splitter': ['best', 'random']},
             return_train_score=True, scoring='accuracy')

- The best hyperparameters were obtained from the cross-validated grid search process. The resulting best_params_ are as follows:

  - criterion: gini
  - max_depth: None
  - max_features: None
  - min_samples_leaf: 4
  - min_samples_split: 20
  - splitter: best

In [26]:
model_classifier_prepruning_v1.best_params_

{'criterion': 'gini',
 'max_depth': None,
 'max_features': None,
 'min_samples_leaf': 4,
 'min_samples_split': 20,
 'splitter': 'best'}

- The mean_train_score is 0.907917, representing the average model performance on the training subsets within each cross-validation fold.

- The mean_test_score is 0.83750, corresponding to the average performance on the validation subsets in each fold.

- The difference between these two values is approximately 0.07, indicating a decrease in performance when moving from training data used for fitting to unseen validation data within each fold.

- Overall, the model performs better on the data used for fitting than on the validation data, although the gap is not large.

In [27]:
scores = pd.DataFrame(model_classifier_prepruning_v1.cv_results_)
scores.sort_values(by="mean_test_score", ascending=False)[['mean_train_score', 'mean_test_score', 'params']]

,mean_train_score,mean_test_score,params
662,0.907917,0.83750,"{'criterion': 'gini', 'max_depth': 20, 'max_fe..."
22,0.907917,0.83750,"{'criterion': 'gini', 'max_depth': None, 'max_..."
822,0.907917,0.83750,"{'criterion': 'gini', 'max_depth': 30, 'max_fe..."
502,0.907917,0.83500,"{'criterion': 'gini', 'max_depth': 10, 'max_fe..."
2548,0.936806,0.83000,"{'criterion': 'log_loss', 'max_depth': 10, 'ma..."
...,...,...,...
1369,0.591528,0.55375,"{'criterion': 'entropy', 'max_depth': 5, 'max_..."
2297,0.591528,0.55375,"{'criterion': 'log_loss', 'max_depth': 5, 'max..."
2301,0.591528,0.55375,"{'criterion': 'log_loss', 'max_depth': 5, 'max..."
1373,0.591528,0.55375,"{'criterion': 'entropy', 'max_depth': 5, 'max_..."


In [28]:
model_classifier_prepruning_v1_best = model_classifier_prepruning_v1.best_estimator_
print ( f"Profundidad del árbol: {model_classifier_prepruning_v1_best.get_depth()} " )
print ( f"Número de nodos terminales: {model_classifier_prepruning_v1_best.get_n_leaves()} " )

Profundidad del árbol: 12 
Número de nodos terminales: 41 


In [29]:
print(f'The best score: {model_classifier_prepruning_v1.best_score_:,.2f}')
print(f'The best params: {model_classifier_prepruning_v1.best_params_}')

The best score: 0.84
The best params: {'criterion': 'gini', 'max_depth': None, 'max_features': None, 'min_samples_leaf': 4, 'min_samples_split': 20, 'splitter': 'best'}


In [30]:
y_predict_prepruning_v1_test = model_classifier_prepruning_v1_best.predict(X_test)
y_predict_prepruning_v1_train = model_classifier_prepruning_v1_best.predict(X_train)

The model achieved very high performance on the training data, reaching approximately 90.6% accuracy. This indicates that the algorithm learned the patterns present in the training dataset very effectively.

However, when evaluated on the test data which represents new information not seen during training the performance dropped to 74% accuracy.

This suggests that the model learned the training data too well, including specific patterns or noise, and therefore does not generalize as effectively to new data, although it still performs moderately well.

In [31]:
print(f'Prediction Data Train: {accuracy_score(y_train, y_predict_prepruning_v1_train)}')
print(f'Prediction Data Test: {accuracy_score(y_test, y_predict_prepruning_v1_test)}')

Prediction Data Train: 0.90625
Prediction Data Test: 0.74


# **Utils**
---

**@By**: Steven Bernal

**@Nickname**: Kaiziferr

**@Git**: https://github.com/Kaiziferr